# 4. On-Behalf-Of (OBO) — preserving user identity across tiers

**Scenario**: a user Alice signs into a frontend, which calls API-A (middle tier). API-A needs to call API-B, but API-B must still know that **Alice** is the one asking — so it can apply her row-level permissions.

Options:

| Option | Who does API-B think is calling? | Row-level security? |
|--------|----------------------------------|---------------------|
| Client credentials | API-A (the app) | No — API-B sees an app, not Alice |
| Forward Alice's token | Alice, but `aud` = API-A, fails validation at API-B | Broken |
| **On-Behalf-Of** | Alice — a fresh token `aud=api://api-b`, `upn=alice@...` | ✅ |

OBO is a special grant type where API-A exchanges Alice's token for a new token targeting API-B, while preserving her identity claims.

## The flow

```
Alice → /proxy/files  (Bearer: alice-token for api://api-a)
         │
         └── API-A → POST /token  grant=jwt-bearer
                      assertion=alice-token
                      scope=api://api-b/Files.Read
                      client_id=api-a  client_secret=...
                      requested_token_use=on_behalf_of
                    ← new token  aud=api://api-b  upn=alice  scp=Files.Read
         │
         └── GET http://api-b/files   (Bearer: new token)
                     → returns only Alice's files
```

Key points:

- API-A proves **itself** to Entra (client_id + secret) — Entra won't let any app swap tokens.
- The `assertion` is the user's original token. Entra validates it was issued for API-A.
- Entra returns a **new** token with a different `aud` but the same user identity.
- This requires that API-A has been granted the `Files.Read` delegated permission on api-b, *and* the user (or an admin) has consented.

In [ ]:
import httpx, json, base64

TOKEN_URL = 'http://localhost:9100/contoso/oauth2/v2.0/token'
API_A     = 'http://localhost:8001'
API_B     = 'http://localhost:8002'

# ⛔ INSPECTION ONLY: no signature check, no claim validation (see notebook 1).
def decode(t):
    p = t.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(p + '=' * (-len(p) % 4)))

# --- Step 1: Alice signs in (ROPC - demo only; production uses auth code + PKCE) ---
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'password',
    'client_id': 'api-a-client-id',
    'client_secret': 'api-a-secret-value',
    'username': 'alice@contoso.com',
    'password': 'alice-password',
    'scope': 'api://api-a/access_as_user',
})
r.raise_for_status()
alice_token = r.json()['access_token']
alice_claims = decode(alice_token)
print('Alice\'s token (aud=api-a):')
print(json.dumps(alice_claims, indent=2))

# This token is for the MIDDLE TIER, not for api-b. That is the whole reason OBO exists.
assert alice_claims['aud'] == 'api://api-a', f"expected aud=api://api-a, got {alice_claims['aud']}"
assert alice_claims['upn'] == 'alice@contoso.com'
assert alice_claims['scp'] == 'access_as_user', f"expected a delegated scope, got {alice_claims.get('scp')}"
assert 'roles' not in alice_claims, 'a delegated token carries scp, not roles'

# Proof that "just forward Alice's token to api-b" is broken, not merely impolite:
direct = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {alice_token}'})
print('\nforwarding the raw user token to api-b ->', direct.status_code)
assert direct.status_code == 401, (
    f'api-b must reject a token minted for api-a (wrong aud); got {direct.status_code}'
)

> ## ⚠️ ROPC (`grant_type=password`) is a teaching crutch, not a pattern
>
> We use it because a notebook cannot perform a browser redirect, and we need *some* user
> token to feed the OBO exchange. In production Alice's token comes from the
> **authorization code flow with PKCE**, and nothing else:
>
> - the browser/app redirects Alice to `/authorize`, she authenticates with Entra directly,
>   and your code **never touches her password**;
> - PKCE (RFC 7636) binds the returned code to the process that started the request, so an
>   intercepted code is useless — mandatory for SPAs and mobile apps, recommended everywhere.
>
> ROPC cannot do any of that. Because your app handles the raw password, it is incompatible
> with MFA, federated/guest accounts, Conditional Access and passwordless sign-in; Entra
> blocks it outright for personal Microsoft accounts and for any account with MFA enforced,
> and OAuth 2.1 removes the grant entirely. The **implicit flow** — the other thing you may
> see in old samples for getting a browser token — is likewise deprecated by RFC 9700 and
> removed in OAuth 2.1. Authorization code + PKCE replaces both.

In [ ]:
# --- Step 2: Alice calls API-A ---
r = httpx.get(f'{API_A}/proxy/files', headers={'Authorization': f'Bearer {alice_token}'})
assert r.status_code == 200, f'expected 200 from api-a, got {r.status_code}: {r.text}'
body = r.json()
print(json.dumps(body, indent=2))

# The lesson of this notebook, asserted: Alice's identity survived the hop, and api-b
# used it to filter rows.
assert body['api_a_saw_user'] == 'alice@contoso.com'
downstream_view = body['api_b_response']
assert downstream_view['mode'] == 'delegated', f"expected delegated, got {downstream_view['mode']}"
assert downstream_view['user'] == 'alice@contoso.com', 'api-b must still see Alice, not api-a'
assert downstream_view['caller_app'] == 'api-a-client-id', 'the calling app is api-a'
names = sorted(f['name'] for f in downstream_view['files'])
assert names == ['budget.xlsx', 'design-doc.md'], f'expected only Alice\'s two files, got {names}'
assert not any(f['owner'] != 'alice@contoso.com' for f in downstream_view['files']), (
    "bob's file leaked through the OBO hop - row-level security is not being applied"
)

Look at the response:
- `api_a_saw_user` = `alice@contoso.com` (API-A validated her token)
- Inside `api_b_response`:
  - `mode: delegated` (API-B saw a user token, not app-only)
  - `user: alice@contoso.com` (preserved via OBO)
  - Only Alice's files — bob's are filtered out

## What API-A actually did

Inspect [`api-a/server.py`](../api-a/server.py) — specifically `_exchange_obo`. Or reproduce it by hand:

In [ ]:
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'urn:ietf:params:oauth:grant-type:jwt-bearer',
    'client_id': 'api-a-client-id',
    'client_secret': 'api-a-secret-value',
    'assertion': alice_token,
    'scope': 'api://api-b/Files.Read',
    'requested_token_use': 'on_behalf_of',
})
r.raise_for_status()
downstream = r.json()['access_token']
d = decode(downstream)
print('Downstream token (aud=api-b, user=alice):')
print(json.dumps(d, indent=2))

# Same human, different audience and different scope. Both halves matter.
assert d['oid'] == alice_claims['oid'] and d['upn'] == alice_claims['upn'], 'identity must be preserved'
assert d['aud'] == 'api://api-b' != alice_claims['aud'], 'the audience must have been re-targeted'
assert d['scp'] == 'Files.Read' != alice_claims['scp'], 'the downstream scope replaces the inbound one'
assert 'roles' not in d, 'OBO yields a delegated token - roles would mean the user vanished'

# And the middle tier really does have to prove itself: no secret, no exchange.
bad = httpx.post(TOKEN_URL, data={
    'grant_type': 'urn:ietf:params:oauth:grant-type:jwt-bearer',
    'client_id': 'api-a-client-id',
    'client_secret': 'WRONG',
    'assertion': alice_token,
    'scope': 'api://api-b/Files.Read',
    'requested_token_use': 'on_behalf_of',
})
print('\nOBO with a bad client secret ->', bad.status_code)
assert bad.status_code == 401, (
    f'holding a user token must not be enough to exchange it; got {bad.status_code}'
)

Compare with Alice's original token — same `oid`, same `upn`, **different `aud`** and **different `scp`**.

## Compare to client credentials

Same user hits API-A, but API-A uses client credentials instead:

In [ ]:
r = httpx.get(f'{API_A}/proxy/files/admin', headers={'Authorization': f'Bearer {alice_token}'})
assert r.status_code == 200, f'expected 200, got {r.status_code}: {r.text}'
admin_body = r.json()
print(json.dumps(admin_body, indent=2))

# The contrast this section exists to show: Alice asked, but api-b never heard of her.
b = admin_body['api_b_response']
assert b['mode'] == 'app-only', f"expected app-only, got {b['mode']}"
assert b.get('user') is None, 'a client-credentials hop erases the user - that is the point'
assert len(b['files']) == 3, f"app-only sees every file, got {len(b['files'])}"
assert len(b['files']) > len(downstream_view['files']), (
    'client credentials must return MORE than the OBO call, otherwise this comparison '
    'demonstrates nothing about row-level security'
)

Now API-B returns **all three** files. `mode: app-only` because the downstream token has `roles`, no `upn`. API-B can't enforce Alice's permissions because it doesn't know who she is.

**Use OBO when you need row-level security or per-user auditing.** Use client credentials for system-level operations (batch jobs, admin tools).

> Note the trap in that sentence. `/proxy/files/admin` validated *Alice's* token and then
> called API-B with API-A's **own** app-only permissions. If API-A doesn't check what Alice
> is allowed to do before making that call, it has just handed every authenticated user the
> union of API-A's application permissions — the classic **confused deputy**. Whenever you
> drop from delegated to app-only, the authorization check has to move *into* the middle
> tier. Our `/proxy/files/admin` deliberately does not do that check; a real one must.

## Common OBO pitfalls

| Error | Cause |
|-------|-------|
| `AADSTS50013: Assertion failed signature validation` | Incoming token was for a *different* audience than your middle tier. It must be `api://api-a`. |
| `AADSTS65001: The user has not consented` | User (or admin) hasn't consented to the downstream scope. Add it to API-A's API permissions. |
| `AADSTS50105: Signed in user not assigned to application role` | Assignment required on the enterprise app — add the user or the group. |
| `AADSTS50076 / AADSTS50079: interaction_required` | **Conditional Access** demands MFA (or a compliant device) for the *downstream* resource and the incoming assertion doesn't satisfy it. See below — this one is not a config typo you can fix once. |
| `AADSTS50173: Token has expired / password changed` | The user's session was revoked. Surface it as a re-auth prompt, not a 500. |
| OBO returns a token but API-B rejects it | Check `aud` in the downstream token matches what API-B validates. |

## Conditional Access & MFA claims — the part that breaks OBO in production

Conditional Access policies are evaluated **per resource**, and OBO crosses a resource
boundary. So a user can hold a perfectly valid token for API-A while the policy on API-B
requires MFA she has not performed. Entra then refuses the exchange with
`interaction_required` and a **`claims` challenge** in the error body.

The middle tier cannot solve this by itself — only the user's browser can, because only the
browser can prompt. The correct handling is a relay:

1. API-A catches the `interaction_required` error and pulls out the opaque `claims` blob.
2. API-A returns **HTTP 401** with `WWW-Authenticate: Bearer error="insufficient_claims",
   claims="<base64 of that blob>"`.
3. The frontend takes the blob and passes it verbatim to MSAL
   (`acquireTokenPopup({ claims })`), which re-authenticates Alice with the extra
   requirement satisfied.
4. The frontend retries with the fresh token; the OBO exchange now succeeds.

Swallowing the error and falling back to client credentials "so the call goes through" is
precisely the confused-deputy bug above, and it silently defeats the CA policy an admin
deliberately configured.

Related claims worth recognising when you decode a *user* token:

| Claim | Meaning |
|-------|---------|
| `amr` | Authentication methods used, e.g. `["pwd", "mfa"]`. **Do not** hand-roll `if "mfa" in amr` as your policy — express it as a Conditional Access policy or an authentication-context requirement, so admins can see and change it. |
| `acrs` | Authentication-context class references the token satisfies (`c1`, `c2`, …). This is the supported way to demand step-up auth for one specific operation. |
| `auth_time` | When the user actually authenticated — for "re-auth within N minutes" rules. |
| `xms_cc` | Signals the client can handle claims challenges; Entra only issues certain CAE claims to clients that advertise it. |

**Continuous Access Evaluation (CAE)** is the other half: with CAE, Entra pushes revocation
events (user disabled, password reset, network location change) to participating resources so
a stolen token dies in minutes instead of at `exp`. Our mock implements none of this — it has
no revocation at all, so **every token it issues is valid for its full hour no matter what**.
That is a property of the mock, not of Entra.

## Configuring OBO in real Entra

1. On **API-B** app registration → *Expose an API* → add scope `Files.Read`.
2. On **API-A** app registration → *API permissions* → add delegated `Files.Read` on API-B. Grant admin consent.
3. On **API-A** → *Expose an API* → add scope `access_as_user`. The frontend requests this when the user signs in.
4. In code — use MSAL: `ConfidentialClientApplication.acquire_token_on_behalf_of(user_assertion=alice_token, scopes=['api://api-b/Files.Read'])`.
   Give MSAL a **per-user** token cache keyed on the assertion's `oid`/`sub`; a shared cache
   across users is a cross-tenant data leak waiting to happen. MSAL's
   `acquire_token_on_behalf_of` does this for you when you keep one app object per process
   and let it manage the cache — never key the cache on the middle tier's own identity.

## Summary

- OBO preserves *user* identity across service hops.
- The middle tier must prove itself with its own creds + present the user's token as the `assertion`.
- Downstream token has `upn`/`scp`, same `oid` as original, **different `aud`** and `scp`.
- Pick OBO when per-user authorization matters; pick client credentials for system tasks —
  and when you pick client credentials, move the authorization check into the middle tier.
- Conditional Access can fail the exchange at runtime. Relay the claims challenge; never
  downgrade to app-only to route around it.